In [20]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import pytz

In [21]:
# Load Playstore dataset
apps = pd.read_csv(r"C:\Users\vanda\Downloads\Play Store Data.csv")
apps.head()  

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [22]:
# Clean Installs
apps["Installs"] =(apps["Installs"].astype(str).str.replace(",","",regex = False).str.replace("+","",regex = False))

In [23]:
apps["Installs"] = pd.to_numeric(apps["Installs"],errors = "coerce")

In [24]:
# Clean Price
apps["Price"] = (apps["Price"].astype(str).str.replace("$","",regex = False))

In [25]:
apps["Price"] = pd.to_numeric(apps["Price"],errors = "coerce").fillna(0)

In [26]:
# Revenue = Installs  * price
apps["Revenue"] = apps["Installs"] * apps["Price"]

In [27]:
# Convert size to MB
apps["Size"] = (apps["Size"].astype(str).str.replace("M","",regex = False))

In [28]:
apps["Size"] = pd.to_numeric(apps["Size"],errors = "coerce")

In [29]:
# Android Version
apps["Android Ver"] = (apps["Android Ver"].astype(str).str.extract(r'(\d+\.\d+)')[0])

In [30]:
apps["Android Ver"] = pd.to_numeric(apps["Android Ver"],errors = "coerce")

In [31]:
# Apply Filters
apps = apps[
    (apps["Installs"] >=10000) &
    (apps["Revenue"] >=10000) &
    (apps["Android Ver"] > 4.0) &
    (apps["Size"] > 15) &
    (apps["Content Rating"] == "Everyone") &
    (apps["App"].str.len() <= 30)
    ]

In [32]:
# Top 3 Categories by Installs
top_categories = (apps.groupby("Category")["Installs"].sum().nlargest(3).index)

apps = apps[apps["Category"].isin(top_categories)]

In [33]:
# Group Data
summary = (apps.groupby(["Category","Type"]).agg(
        Average_Installs = ("Installs","mean"),
        Revenue = ("Revenue","mean")
).reset_index()
)
summary["Group"] = summary["Category"] + "-" + summary["Type"]
                    

In [34]:
# Time Restriction
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

In [35]:
if current_time >= datetime.strptime("13:00","%H:%M").time() and current_time <= datetime.strptime("14:00","%H:%M").time():
    
# Dual Axis Chart
    fig = go.Figure()

# Average Installs(Bar)
    fig.add_trace(
        go.Bar(
            x = summary["Group"],
            y = summary["Average_Installs"],
            name = "Average Installs"
        )
    )

# Revenue(Line)
    fig.add_trace(
        go.Scatter(
            x = summary["Group"],
            y = summary["Revenue"],
            mode = "lines+markers",
            name = "Average Revenue",
            yaxis = "y2"
        )
    )
    fig.update_layout(
        title = "Average Installs vs Revenue (Free vs Paid Apps)",
        xaxis_title = "Category and App Type",
         yaxis = dict(
             title = "Average Installs"
         ),
        yaxis2 = dict(
            title = "Average Revenue($)",
            overlaying = "y",
            side = "right"
        ),
        barmode = "group",
        template = "plotly_white",
        height = 600
    )
    
    fig.show()

else:
    print("Dual-axis Chart is only available between 1 PM IST and 2 PM IST")


    

Dual-axis Chart is only available between 1 PM IST and 2 PM IST


In [36]:
#Conclusion:
#The dual-axis chart compares the average installs (bars) and average revenue (line) for the top three app categories after applying all required filters.
#The visualization shows that the Photography category has the highest average installs and revenue among the filtered apps. 
#The dashboard is configured to display this chart only between 1:00 PM and 2:00 PM IST, meeting all assignment requirements.
